In [1]:
using Glob 
using DelimitedFiles, DataFrames, EzXML
using Dates
using LinearAlgebra
using ForwardDiff
using Statistics
using Interpolations
using Rotations
using Unitful
using ThreadsX
using WENO4
using Polyester
using CoordinateTransformations
import PhysicalConstants.CODATA2018: m_p, m_e, e, ε_0, μ_0, c_0, G, k_B
const M_Mars = 0.64169e24u"kg"
const R_Mars = 3389.5u"km"  #m
const au = 149597870.7u"km"

include("../../src/scripts/MAVEN_load.jl")
include("../../src/scripts/MAVEN_STATIC.jl")
include("../../src/scripts/SphericalSplines.jl")
using .MAVEN_STATIC
using .MAVEN_load
using .SphericalSplines

date = Date(2022, 8, 24)
datestr = Dates.format.(date, "yyyymmdd")
year_num = Dates.format(date, "yyyy")
month_num = Dates.format(date, "mm")

ts = DateTime(date) + Dates.Hour(7) + Dates.Minute(27)
te = ts + Dates.Minute(1)

data_dir = "E:\\Data"
output_dir = "E:\\Output"
MAVEN_path = joinpath(data_dir, "MAVEN")
MAG_path = joinpath(MAVEN_path, "MAG", "l2", year_num, month_num)
STATIC_path = joinpath(MAVEN_path, "STATIC", "l2", year_num, month_num)
MAVEN_output_path = joinpath(output_dir, "MAVEN")
STATIC_output_path = joinpath(MAVEN_output_path, "STATIC")
VDF_path = joinpath(STATIC_output_path, "VDF", year_num, month_num)
mkpath(VDF_path)
MAG_file = glob("mvn_mag_l2*$datestr*.sts", MAG_path)
STA_file = glob("mvn_sta_l2*d1*32e4d16a8m*$datestr*.cdf", STATIC_path)
println("MAG file: ", MAG_file)
println("STATIC file: ", STA_file)
println("Output path: ", VDF_path)

MVN_mag2c1hz = MAVEN_load.load_mag_l2(MAG_file[1])
MVN_static = MAVEN_load.load_cdf(STA_file[1])
println("Data loaded.")

MAG file: ["E:\\Data\\MAVEN\\MAG\\l2\\2022\\08\\mvn_mag_l2_2022236ss_20220824_v01_r01.sts"]
STATIC file: ["E:\\Data\\MAVEN\\STATIC\\l2\\2022\\08\\mvn_sta_l2_d1-32e4d16a8m_20220824_v02_r03.cdf"]
Output path: E:\Output\MAVEN\STATIC\VDF\2022\08
Data loaded.


In [ ]:
# MVN MAG
datejul = datetime2julian(DateTime(date))
MVN_magut1hz = MVN_mag2c1hz[:JulUTtime].-datejul
MVN_BMSO1hz = MVN_mag2c1hz[:B]
ind = ThreadsX.findall(ts .<= MVN_mag2c1hz[:epoch] .<= te)
MVN_Bmeanmso = dropdims(mean(Matrix(MVN_BMSO1hz[ind, :]), dims=(1,)), dims=(1,))

# MVN STATIC
MVN_staticswp = MVN_static[:swp_ind] |> Array 
MVN_staticswp .+= 1
MVN_statictime = MVN_static[:epoch] |> Array
MVN_staticjulian=datetime2julian.(DateTime.(MVN_statictime))
MVN_staticeflux = MVN_static[:eflux] |> Array
MVN_staticmass = MVN_static[:mass_arr] |> Array
MVN_staticenergy = MVN_static[:energy] |> Array
MVN_statictheta = MVN_static[:theta] |> Array
MVN_staticphi = MVN_static[:phi] |> Array
MVN_staticdenergy = MVN_static[:denergy] |> Array
MVN_staticdtheta = MVN_static[:dtheta] |> Array
MVN_staticdphi = MVN_static[:dphi] |> Array
MVN_staticdomega = MVN_static[:domega] |> Array
MVN_staticvelocity = sqrt.(MVN_staticenergy.*(2.0/ustrip(u"eV", m_p*c_0^2))./MVN_staticmass).*ustrip(u"km/s", c_0)
MVN_staticpos_sc_mso = MVN_static[:pos_sc_mso] |> Array
MVN_staticpos_sc_mso /= ustrip(u"km", R_Mars)

MVN_staticquat_mso = MVN_static[:quat_mso] |> Array
MVN_staticquat_sc = MVN_static[:quat_sc] |> Array
MVN_staticmsotrans = [QuatRotation(quat) for quat in eachcol(MVN_staticquat_mso')]
MVN_staticind = ThreadsX.findall(ts .<= MVN_statictime .<= te)
MVN_staticswpind = MVN_staticswp[MVN_staticind[1]]
MVN_staticefluxmean = dropdims(mean(MVN_staticeflux[MVN_staticind, :, :, :], dims = (1, ) ), dims=(1, ))
MVN_staticpsdmean = ustrip(u"cm^-3 * km^-3 * s^3", 0.5 * 1u"cm^-2 * s^-1" * (m_p/1u"eV")^2)  .* MVN_staticefluxmean .* (MVN_staticmass[:,:,:,MVN_staticswpind,1]./ MVN_staticenergy[:,:,:,MVN_staticswpind,1]).^2

MVN_staticindmean = round(Int64, mean(MVN_staticind))
MVN_statictimemean = MVN_statictime[MVN_staticindmean]
MVN_staticjulianmean = MVN_staticjulian[MVN_staticindmean]
MVN_static2msomean = QuatRotation(MVN_staticquat_mso[MVN_staticindmean, :])
MVN_Bmeanstatic = MVN_static2msomean' * MVN_Bmeanmso
MVN_parastatic = MVN_Bmeanstatic ./ norm(MVN_Bmeanstatic)
swi, swj = 2, 12
ind = argmax(MVN_staticefluxmean[1, (swj-1)*4+swi, :] ./ MVN_staticenergy[1, 1, :, MVN_staticswpind, 1].^2)
MVN_staticvswrough = CartesianFromSpherical()(CoordinateTransformations.Spherical(
    MVN_staticvelocity[1, (swj-1)*4+swi, ind, MVN_staticswpind, 1], 
    deg2rad(MVN_staticphi[1, (swj-1)*4+swi, ind, MVN_staticswpind, 1]), 
    deg2rad(MVN_statictheta[1, (swj-1)*4+swi, ind, MVN_staticswpind, 1])))
MVN_staticdirections = [CartesianFromSpherical()(CoordinateTransformations.Spherical(1.0, deg2rad(MVN_staticphi[1, ij, ind, MVN_staticswpind, 1]), deg2rad(MVN_statictheta[1, ij, ind, MVN_staticswpind, 1]))) for ij in 1:64]
MVN_staticss = []
Snan = SphericalSplines.InterpolationSpline(stack(MVN_staticdirections; dims=1), fill(-30.0, 64))
for k in 1:32
    ind = findall(MVN_staticefluxmean[1, :, k] .> 0.0)
    if !isempty(ind)
        S = SphericalSplines.InterpolationSpline(stack(MVN_staticdirections[ind]; dims=1), log10.(MVN_staticefluxmean[1, ind, k]))
        push!(MVN_staticss, S)
    else
        push!(MVN_staticss, Snan)
    end  
end
MVN_perpstatic1 = cross(MVN_parastatic, MVN_staticvswrough)
MVN_perpstatic1 = MVN_perpstatic1 ./ norm(MVN_perpstatic1)
MVN_perpstatic2 = cross(MVN_perpstatic1, MVN_parastatic)
MVN_static2mfa = transpose(hcat(MVN_perpstatic1, MVN_parastatic, MVN_perpstatic2))

# 预计算有效能道（排除Snan）
valid_channels = [k for k in 1:32 if MVN_staticss[k] !== Snan]
valid_logv = log10.(MVN_staticvelocity[1, 1, valid_channels, MVN_staticswpind, 1])
println("Valid WENO4 channels: $(length(valid_channels))/27")

3×3 StaticArraysCore.SMatrix{3, 3, Float64, 9} with indices SOneTo(3)×SOneTo(3):
  0.198877   0.194827  -0.960464
 -0.904946  -0.339692  -0.256287
 -0.376193   0.920137   0.108751

In [29]:
function nanmean(arr::AbstractArray{T,3}, dim::Int) where T
    # Count non-NaN along dim
    count_valid = sum(.!isnan.(arr), dims=dim)
    # Sum ignoring NaN
    sum_valid = sum(ifelse.(isnan.(arr), zero(T), arr), dims=dim)
    # Divide, keeping NaN where all were NaN
    result = sum_valid ./ count_valid
    result[count_valid .== 0] .= NaN
    return dropdims(result, dims=dim)
end

# Usage: average along dimension 3
result = nanmean(MVN_static_eflux_rect, 1)

using CairoMakie
CairoMakie.activate!()

fig = Figure(size = (800, 700))
ax = Axis(fig[1, 1], 
    xlabel = "X", 
    ylabel = "Y",
    aspect = 1)  # square cells

hm = heatmap!(ax, vperp1, vpara, result,
    colormap = :gist_earth,    # same as codebase
    colorscale = log10,        # log scale (common for flux)
    colorrange = (1e4, 1e8),   # adjust to your data range
    nan_color = :transparent   # NaN shows as transparent
)

Colorbar(fig[1, 2], hm, label = "Flux")
save("heatmap.png", fig)

In [26]:
save("heatmap.png", fig)